# 🎤 Speaker Verification with Pre-trained X-Vector Fine-Tuning on Kaggle (2x T4 GPUs)

This notebook implements the end-to-end pipeline for **fine-tuning a pre-trained X-Vector model** (from SpeechBrain) for Speaker Verification.

### Pipeline Overview:
1. **Environment Preparation**: Setup dependencies (including SpeechBrain) and configure dual GPU training.
2. **Data Pipeline & Augmentation**: Scan VoxCeleb and MUSAN datasets recursively. The data loader returns raw audio waveforms, which are preprocessed by the pre-trained feature extractor on the GPU.
3. **Model Architecture**: Load pre-trained SpeechBrain X-Vector modules and wrap them inside a custom PyTorch model for fine-tuning.
4. **Training & Validation**: Fine-tune the backbone and optimize the similarity threshold on validation pairs to maximize **F1-score**.
5. **Visualization & Stress Testing**: Plot training curves, run stress testing under fixed noise SNRs (0dB, 10dB, 20dB), and plot ROC curves.
6. **Export**: Export the fine-tuned model and evaluation metadata.

## 🛠️ Step 1: Prepare Environment and Import Libraries

In [ ]:
# Install dependencies including speechbrain
!pip install -q speechbrain torchaudio soundfile librosa matplotlib seaborn scikit-learn pandas numpy tqdm

import os, re, sys, time, math, random, glob, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_curve, auc
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f'[INFO] Device: {device} | GPUs: {num_gpus}')
for i in range(num_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
os.makedirs('checkpoints/xvector_finetune', exist_ok=True)
os.makedirs('results', exist_ok=True)

## 🔄 Step 2: Auto-Discover Datasets, Preprocess, and Setup Augmentation

In [ ]:
import warnings
warnings.filterwarnings('ignore')

SR = 16000
DURATION = 3
NUM_SAMPLES = SR * DURATION
BATCH_SIZE = 64
NUM_WORKERS = 4

# ============================================================
# AUTO-DISCOVER DATASET PATHS
# ============================================================
INPUT_ROOT = '/kaggle/input'

print('=' * 60)
print('STEP 1: Discovering /kaggle/input/ structure')
print('=' * 60)
for ds in sorted(os.listdir(INPUT_ROOT)):
    ds_path = os.path.join(INPUT_ROOT, ds)
    if os.path.isdir(ds_path):
        print(f'\n  [DIR] {ds}/')
        for sub in sorted(os.listdir(ds_path))[:15]:
            sub_path = os.path.join(ds_path, sub)
            tag = '[DIR]' if os.path.isdir(sub_path) else '[FILE]'
            print(f'    {tag} {sub}')
            if os.path.isdir(sub_path):
                for sub2 in sorted(os.listdir(sub_path))[:8]:
                    sub2_path = os.path.join(sub_path, sub2)
                    tag2 = '[DIR]' if os.path.isdir(sub2_path) else '[FILE]'
                    print(f'      {tag2} {sub2}')

print(f'\n{"=" * 60}')
print('STEP 2: Searching for all audio files...')
print('=' * 60)
all_wav = glob.glob(os.path.join(INPUT_ROOT, '**', '*.wav'), recursive=True)
all_flac = glob.glob(os.path.join(INPUT_ROOT, '**', '*.flac'), recursive=True)
print(f'  .wav  files found: {len(all_wav)}')
print(f'  .flac files found: {len(all_flac)}')
all_audio = all_wav + all_flac
print(f'  Total audio files: {len(all_audio)}')

if len(all_audio) > 0:
    print('\n  Sample paths:')
    for p in all_audio[:10]:
        print(f'    {p}')

# ============================================================
# CLASSIFY: VoxCeleb vs MUSAN
# ============================================================
print(f'\n{"=" * 60}')
print('STEP 3: Classifying audio files (VoxCeleb vs MUSAN)')
print('=' * 60)

vox_wav_files = []
noise_files = []

for f in all_audio:
    fp = f.replace('\\', '/')
    if re.search(r'/id\d{3,}/', fp):
        vox_wav_files.append(f)
    elif 'noise' in fp.lower() and 'musan' in fp.lower():
        noise_files.append(f)

if len(vox_wav_files) == 0:
    print('  [WARN] No VoxCeleb files found by id pattern. Trying broader match...')
    for f in all_audio:
        fp = f.replace('\\', '/').lower()
        if 'vox' in fp or 'celeb' in fp:
            vox_wav_files.append(f)

if len(noise_files) == 0:
    print('  [WARN] No MUSAN noise files found. Trying broader match...')
    for f in all_audio:
        fp = f.replace('\\', '/').lower()
        if 'noise' in fp:
            noise_files.append(f)

print(f'  VoxCeleb audio files: {len(vox_wav_files)}')
print(f'  MUSAN noise files:   {len(noise_files)}')

assert len(vox_wav_files) > 0, 'ERROR: No VoxCeleb audio files found!'

# ============================================================
# BUILD SPEAKER DICTIONARY
# ============================================================
def get_speaker_id(path):
    parts = path.replace('\\', '/').split('/')
    for p in parts:
        if re.match(r'^id\d{3,}$', p):
            return p
    return 'unknown'

speaker_dict = {}
for path in vox_wav_files:
    spk_id = get_speaker_id(path)
    if spk_id != 'unknown':
        speaker_dict.setdefault(spk_id, []).append(path)

if len(speaker_dict) == 0:
    print('  [WARN] No idXXXXX folders found. Using parent directories as speaker labels.')
    for path in vox_wav_files:
        parts = path.replace('\\', '/').split('/')
        if len(parts) >= 3:
            spk_id = parts[-3]
            speaker_dict.setdefault(spk_id, []).append(path)

print(f'  Total unique speakers: {len(speaker_dict)}')

# Scale training setup (None = train on all speakers)
SELECT_SUBSET_SPEAKERS = None
if SELECT_SUBSET_SPEAKERS is not None and len(speaker_dict) > SELECT_SUBSET_SPEAKERS:
    sorted_spk = sorted(speaker_dict.items(), key=lambda x: len(x[1]), reverse=True)
    speaker_dict = dict(sorted_spk[:SELECT_SUBSET_SPEAKERS])
    vox_wav_files = [f for files in speaker_dict.values() for f in files]
    print(f'  Selected top {SELECT_SUBSET_SPEAKERS} speakers ({len(vox_wav_files)} files)')

speakers = sorted(speaker_dict.keys())
spk_to_label = {spk: idx for idx, spk in enumerate(speakers)}
label_to_spk = {idx: spk for idx, spk in enumerate(speakers)}
NUM_CLASSES = len(speakers)
print(f'  NUM_CLASSES = {NUM_CLASSES}')

# ============================================================
# NOISE AUGMENTATION
# ============================================================
class AudioAugmentation:
    def __init__(self, noise_files, sr=16000):
        self.noise_files = noise_files
        self.sr = sr
    def add_noise(self, wave, snr_db=None):
        if len(self.noise_files) == 0: return wave
        if snr_db is None: snr_db = random.uniform(0.0, 15.0)
        try:
            nw, _ = librosa.load(random.choice(self.noise_files), sr=self.sr)
        except Exception:
            return wave
        if len(nw) < len(wave):
            nw = np.tile(nw, math.ceil(len(wave)/len(nw)))[:len(wave)]
        else:
            s = random.randint(0, len(nw)-len(wave))
            nw = nw[s:s+len(wave)]
        cp = np.mean(wave**2) + 1e-8
        np_ = np.mean(nw**2) + 1e-8
        scale = math.sqrt(cp / np_ * 10**(-snr_db/10.0))
        mixed = wave + scale * nw
        mx = np.max(np.abs(mixed))
        return mixed / mx if mx > 1.0 else mixed

augmenter = AudioAugmentation(noise_files, SR)

# ============================================================
# TRAIN/VAL SPLIT
# ============================================================
train_files, val_files = [], []
for spk, files in speaker_dict.items():
    if len(files) >= 4:
        tr, va = train_test_split(files, test_size=0.25, random_state=42)
        train_files.extend([(f, spk) for f in tr])
        val_files.extend([(f, spk) for f in va])
    else:
        train_files.extend([(f, spk) for f in files])
        val_files.extend([(f, spk) for f in files])
print(f'  Train samples: {len(train_files)} | Val samples: {len(val_files)}')

# ============================================================
# PYTORCH DATASET (LAZY LOADING RAW AUDIO)
# ============================================================
class VoxCelebRawDataset(Dataset):
    def __init__(self, pairs, sr=16000, dur=3, augment=False, augmenter=None):
        self.pairs = pairs
        self.sr = sr
        self.ns = sr * dur
        self.augment = augment
        self.augmenter = augmenter
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        fp, spk = self.pairs[idx]
        label = spk_to_label[spk]
        try:
            wave, _ = librosa.load(fp, sr=self.sr)
        except Exception:
            wave = np.zeros(self.ns, dtype=np.float32)
        if len(wave) < self.ns:
            wave = np.pad(wave, (0, self.ns - len(wave)))
        else:
            s = random.randint(0, len(wave)-self.ns) if self.augment else (len(wave)-self.ns)//2
            wave = wave[s:s+self.ns]
        if self.augment and self.augmenter:
            if random.random() < 0.6:
                wave = self.augmenter.add_noise(wave)
        return torch.tensor(wave, dtype=torch.float32), label

train_dataset = VoxCelebRawDataset(train_files, SR, DURATION, augment=True, augmenter=augmenter)
val_dataset = VoxCelebRawDataset(val_files, SR, DURATION, augment=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f'  DataLoaders ready. Train: {len(train_loader)} | Val: {len(val_loader)}')

def generate_verification_pairs(files_list, num_pairs=1000, seed=42):
    random.seed(seed); pairs = []; d = {}
    for f, s in files_list: d.setdefault(s, []).append(f)
    spks = list(d.keys())
    pos_count = num_pairs // 2
    att = 0
    while len(pairs) < pos_count and att < 10000:
        att += 1; s = random.choice(spks)
        if len(d[s]) >= 2:
            a, b = random.sample(d[s], 2)
            pairs.append((a, b, 1))
    att = 0
    while len(pairs) < num_pairs and att < 10000:
        att += 1; s1, s2 = random.sample(spks, 2)
        pairs.append((random.choice(d[s1]), random.choice(d[s2]), 0))
    return pairs

val_pairs = generate_verification_pairs(val_files, 1000, seed=42)
test_pairs = generate_verification_pairs(val_files, 1000, seed=999)
print(f'  Val pairs: {len(val_pairs)} | Test pairs: {len(test_pairs)}')

## 🧠 Step 3: Pre-trained X-Vector Fine-Tuning Wrapper
We load the SpeechBrain pretrained X-Vector model and expose its sub-components (`compute_features`, `mean_var_norm`, and `embedding_model`) directly as a standard PyTorch trainable module, mapping its 512-dimensional embeddings to our classification head.

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

print("Loading pretrained X-Vector weights from Hugging Face...")
# Note: Requires internet access enabled inside Kaggle first-time to download. 
# Subsequent runs will use the cached checkpoints.
sb_classifier = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", run_opts={"device": device})

class PretrainedXVectorWrapper(nn.Module):
    def __init__(self, classifier, num_classes=10, embedding_dim=512):
        super().__init__()
        # Extract modules from SpeechBrain
        self.feat_extractor = classifier.mods.compute_features
        self.mean_var_norm = classifier.mods.mean_var_norm
        self.embedding_model = classifier.mods.embedding_model
        
        # Custom classification head
        self.fc = nn.Linear(embedding_dim, num_classes)
        
    def extract_embedding(self, wavs):
        # wavs shape: [batch, samples]
        wav_lens = torch.ones(wavs.shape[0], device=wavs.device)
        
        # Differentiable forward pass through pre-trained modules
        feats = self.feat_extractor(wavs)
        feats = self.mean_var_norm(feats, wav_lens)
        embeddings = self.embedding_model(feats, wav_lens)
        return embeddings.squeeze(1)
        
    def forward(self, wavs):
        embeddings = self.extract_embedding(wavs)
        return self.fc(F.relu(embeddings))

model = PretrainedXVectorWrapper(sb_classifier, NUM_CLASSES, 512)

# Enable training of all parameters (backbone + head) for fine-tuning
for param in model.parameters():
    param.requires_grad = True

if num_gpus > 1:
    model = nn.DataParallel(model)
model = model.to(device)

print(f"X-Vector Fine-Tuning wrapper initialized. Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 📉 Step 4: Training & Verification (F1-Score Optimization)
Fine-tune the model using a lower learning rate (typical for transfer learning) and Cosine Annealing.

In [ ]:
EPOCHS = 10  # Fine-tuning requires far fewer epochs (e.g. 5-10) to achieve high convergence
criterion = nn.CrossEntropyLoss()
# Lower learning rate for fine-tuning pre-trained backbone
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def get_embedding(mdl, fp, dev):
    mdl.eval()
    try:
        w, _ = librosa.load(fp, sr=16000)
    except Exception:
        w = np.zeros(48000, dtype=np.float32)
    ns = 48000
    if len(w) < ns: w = np.pad(w, (0, ns-len(w)))
    else: w = w[(len(w)-ns)//2:(len(w)-ns)//2+ns]
    wt = torch.tensor(w, dtype=torch.float32).unsqueeze(0).to(dev)
    with torch.no_grad():
        m2 = mdl.module.extract_embedding(wt) if isinstance(mdl, nn.DataParallel) else mdl.extract_embedding(wt)
    return F.normalize(m2, p=2, dim=1).cpu().numpy()[0]

def eval_verification(mdl, pairs, dev):
    mdl.eval()
    sims, labs = [], []
    cache = {}
    for a, b, lb in pairs:
        if a not in cache: cache[a] = get_embedding(mdl, a, dev)
        if b not in cache: cache[b] = get_embedding(mdl, b, dev)
        sims.append(np.dot(cache[a], cache[b]))
        labs.append(lb)
    del cache; gc.collect(); torch.cuda.empty_cache()
    sims, labs = np.array(sims), np.array(labs)
    best_th, best_f1 = 0.0, 0.0
    for th in np.linspace(-1, 1, 200):
        f = f1_score(labs, (sims>=th).astype(int))
        if f > best_f1: best_f1, best_th = f, th
    preds = (sims >= best_th).astype(int)
    fpr, tpr, _ = roc_curve(labs, sims)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    return {'threshold': best_th, 'f1': best_f1,
            'precision': precision_score(labs, preds, zero_division=0),
            'recall': recall_score(labs, preds, zero_division=0),
            'accuracy': accuracy_score(labs, preds),
            'eer': eer, 'sims': sims, 'labels': labs}

history = {'train_loss':[], 'val_loss':[], 'val_f1':[], 'val_eer':[]}
best_val_f1 = 0.0

for epoch in range(1, EPOCHS+1):
    model.train()
    rl = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for bf, bl in pbar:
        bf, bl = bf.to(device), bl.to(device)
        optimizer.zero_grad()
        logits = model(bf)
        loss = criterion(logits, bl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        rl += loss.item()
        pbar.set_postfix(Loss=f'{loss.item():.4f}')
    etl = rl / len(train_loader)
    
    model.eval()
    vl = 0.0
    with torch.no_grad():
        for bf, bl in val_loader:
            bf, bl = bf.to(device), bl.to(device)
            vl += criterion(model(bf), bl).item()
    evl = vl / len(val_loader)
    
    vr = eval_verification(model, val_pairs, device)
    history['train_loss'].append(etl)
    history['val_loss'].append(evl)
    history['val_f1'].append(vr['f1'])
    history['val_eer'].append(vr['eer'])
    scheduler.step()
    
    print(f'  Loss: {etl:.4f}/{evl:.4f} | F1: {vr["f1"]:.4f} @ {vr["threshold"]:.3f} | EER: {vr["eer"]:.4f}')
    
    if vr['f1'] > best_val_f1:
        best_val_f1 = vr['f1']
        ms = model.module if isinstance(model, nn.DataParallel) else model
        torch.save({'epoch':epoch, 'model_state_dict':ms.state_dict(),
                    'optimizer_state_dict':optimizer.state_dict(),
                    'val_f1':best_val_f1, 'optimal_threshold':vr['threshold'],
                    'spk_to_label':spk_to_label, 'label_to_spk':label_to_spk},
                   'checkpoints/xvector_finetune/best_model.pt', weights_only=False)
        print(f'  * Best F1! Checkpoint saved.')
    
    # Explicit VRAM cleanup after every epoch
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  [MEM] VRAM cache cleared. ', end='')
    if torch.cuda.is_available():
        print(f'Allocated: {torch.cuda.memory_allocated()/1e6:.0f}MB')
    else:
        print()

## 📊 Step 5: Test Set Evaluation & Visualization

In [ ]:
ckpt_path = 'checkpoints/xvector_finetune/best_model.pt'
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    eval_model = PretrainedXVectorWrapper(sb_classifier, NUM_CLASSES, 512)
    eval_model.load_state_dict(ckpt['model_state_dict'])
    eval_model = eval_model.to(device)
    opt_threshold = ckpt['optimal_threshold']
    print(f'[RESTORED] Best model epoch {ckpt["epoch"]} restored. Threshold: {opt_threshold:.3f}')
else:
    eval_model = model
    opt_threshold = 0.5

tr = eval_verification(eval_model, test_pairs, device)
ts, tl = tr['sims'], tr['labels']
tpreds = (ts >= opt_threshold).astype(int)
tf1 = f1_score(tl, tpreds)
tacc = accuracy_score(tl, tpreds)
tprec = precision_score(tl, tpreds)
trec = recall_score(tl, tpreds)
fpr, tpr, _ = roc_curve(tl, ts)
tauc = auc(fpr, tpr)

print(f'\n{"="*50} TEST REPORT {"="*50}')
print(f'  Threshold: {opt_threshold:.3f}')
print(f'  F1: {tf1:.4f} | Accuracy: {tacc:.4f} | Precision: {tprec:.4f} | Recall: {trec:.4f}')
print(f'  EER: {tr["eer"]:.4f} | ROC AUC: {tauc:.4f}')

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
# Loss
axes[0,0].plot(history['train_loss'], label='Train Loss', lw=2)
axes[0,0].plot(history['val_loss'], label='Val Loss', lw=2)
axes[0,0].set_title('CrossEntropy Loss Curve', fontweight='bold'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)
# Verification Metrics
axes[0,1].plot(history['val_f1'], label='F1-Score', color='green', lw=2)
axes[0,1].plot(history['val_eer'], label='EER', color='red', lw=2, ls='--')
axes[0,1].set_title('Validation Speaker Verification Metrics', fontweight='bold'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)
# Score Distributions
sns.histplot(ts[tl==1], color='green', label='Same Speaker (Positive)', kde=True, bins=30, stat='density', alpha=0.4, ax=axes[1,0])
sns.histplot(ts[tl==0], color='red', label='Different Speaker (Negative)', kde=True, bins=30, stat='density', alpha=0.4, ax=axes[1,0])
axes[1,0].axvline(opt_threshold, color='navy', linestyle='--', lw=2, label=f'Threshold={opt_threshold:.3f}')
axes[1,0].set_title('Cosine Similarity Distribution', fontweight='bold'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)
# ROC Curve
axes[1,1].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC (AUC = {tauc:.4f})')
axes[1,1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1,1].set_title('Receiver Operating Characteristic', fontweight='bold'); axes[1,1].legend(loc='lower right'); axes[1,1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/xvector_finetune_evaluation_plots.png', dpi=150)
plt.show()

## 🔊 Step 5.5: Fixed SNR Noisy Audio Evaluation

In [ ]:
def eval_verif_noise(mdl, pairs, dev, snr_val):
    mdl.eval()
    sims, labs = [], []
    cache = {}
    for a, b, lb in tqdm(pairs, desc=f'Evaluating {snr_val}dB'):
        for f in [a, b]:
            if f not in cache:
                try:
                    w, _ = librosa.load(f, sr=16000)
                except Exception:
                    w = np.zeros(48000, dtype=np.float32)
                ns = 48000
                if len(w) < ns: w = np.pad(w, (0, ns-len(w)))
                else: w = w[:ns]
                
                if len(noise_files) > 0:
                    try:
                        nw, _ = librosa.load(random.choice(noise_files), sr=16000)
                        if len(nw) < len(w):
                            nw = np.tile(nw, math.ceil(len(w)/len(nw)))[:len(w)]
                        else:
                            nw = nw[:len(w)]
                        clean_p = np.mean(w**2) + 1e-8
                        noise_p = np.mean(nw**2) + 1e-8
                        scale = math.sqrt(clean_p / noise_p * 10**(-snr_val/10))
                        w = w + scale * nw
                        mx = np.max(np.abs(w))
                        if mx > 1.0: w = w / mx
                    except Exception:
                        pass
                wt = torch.tensor(w, dtype=torch.float32).unsqueeze(0).to(dev)
                with torch.no_grad():
                    m2 = mdl.module.extract_embedding(wt) if isinstance(mdl, nn.DataParallel) else mdl.extract_embedding(wt)
                cache[f] = F.normalize(m2, p=2, dim=1).cpu().numpy()[0]
        sims.append(np.dot(cache[a], cache[b]))
        labs.append(lb)
    del cache; gc.collect(); torch.cuda.empty_cache()
    sims, labs = np.array(sims), np.array(labs)
    preds = (sims >= opt_threshold).astype(int)
    fpr, tpr, _ = roc_curve(labs, sims)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.abs(fpr - fnr))]
    return {'f1': f1_score(labs, preds), 'eer': eer, 'accuracy': accuracy_score(labs, preds),
            'precision': precision_score(labs, preds, zero_division=0), 'recall': recall_score(labs, preds, zero_division=0)}

print('Evaluating Fine-Tuned X-Vector under noise stress test (0dB, 10dB, 20dB)...')
noise_results = {}
for snr in [20, 10, 0]:
    noise_results[snr] = eval_verif_noise(eval_model, test_pairs, device, snr)

print(f'\n{"="*50} NOISE STRESS TEST (FINE-TUNED X-VECTOR) {"="*50}')
print(f'  Clean Base  -> F1: {tf1:.4f} | EER: {tr["eer"]:.4f} | Accuracy: {tacc:.4f} | Precision: {tprec:.4f} | Recall: {trec:.4f}')
for snr in [20, 10, 0]:
    r = noise_results[snr]
    print(f'  SNR {snr:2d} dB   -> F1: {r["f1"]:.4f} | EER: {r["eer"]:.4f} | Accuracy: {r["accuracy"]:.4f} | Precision: {r["precision"]:.4f} | Recall: {r["recall"]:.4f}')


## 💾 Step 6: Export Model for Comparison

In [ ]:
torch.save({
    'model_architecture': 'PretrainedXVectorWrapper',
    'embedding_dim': 512, 'num_classes': NUM_CLASSES,
    'optimal_threshold': opt_threshold,
    'model_state_dict': eval_model.state_dict(),
    'spk_to_label': spk_to_label, 'label_to_spk': label_to_spk,
    'test_metrics': {'f1_score':tf1, 'eer':tr['eer'], 'accuracy':tacc,
                     'precision':tprec, 'recall':trec, 'auc':tauc}
}, 'results/xvector_final_model.pt', weights_only=False)
print('[SUCCESS] Fine-tuned X-Vector exported to results/xvector_final_model.pt')
v = torch.load('results/xvector_final_model.pt', map_location='cpu', weights_only=False)
print(f'  Arch: {v["model_architecture"]} | F1: {v["test_metrics"]["f1_score"]:.4f}')